# 09 - Review Darija GPT evaluation

Notebook 08 trains a checkpoint and writes validation metrics and generated samples to local artifacts. This notebook loads those results, checks that they belong to the same tokenizer, summarizes the held-out scores, and computes simple repetition and n-gram diversity indicators for the generated continuations.

These text indicators are diagnostic signals only. They cannot determine whether Darija is fluent, natural, or factually correct; generated samples still need human review.

## 1. Load the saved evaluation artifacts

Run notebook 08 first. This report reads its JSON metrics and JSONL generation samples; it does not load model weights or repeat training.

In [ ]:
import json
from pathlib import Path

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the darija-llm-lab project directory.")

PROJECT_ROOT = find_project_root()
EVALUATION_DIR = PROJECT_ROOT / "artifacts/evaluation"
EVALUATION_PATH = EVALUATION_DIR / "evaluation.json"
GENERATIONS_PATH = EVALUATION_DIR / "generation_samples.jsonl"
if not EVALUATION_PATH.is_file() or not GENERATIONS_PATH.is_file():
    raise FileNotFoundError(
        "Evaluation artifacts are missing. Run notebook 08 to train and evaluate the model first."
    )

## 2. Create a report object

The report class validates the saved fields, presents loss and perplexity together, and calculates generation indicators from continuations only. This keeps file loading and reporting logic in one reusable class.

In [ ]:
class ModelEvaluationReport:
    def __init__(self, evaluation_path, generations_path):
        self.evaluation_path = Path(evaluation_path)
        self.generations_path = Path(generations_path)
        self.evaluation = json.loads(
            self.evaluation_path.read_text(encoding="utf-8")
        )
        with self.generations_path.open(encoding="utf-8") as stream:
            self.generations = [
                json.loads(line) for line in stream if line.strip()
            ]

        required = {"model_config", "dataset", "metrics", "history", "tokenizer_sha256"}
        missing = required.difference(self.evaluation)
        if missing:
            raise ValueError(f"Evaluation report is missing fields: {sorted(missing)}")
        if not self.generations:
            raise ValueError("No generation samples were saved.")

        for sample in self.generations:
            if not isinstance(sample.get("prompt"), str):
                raise ValueError("Every generation sample must include its prompt.")
            if not isinstance(sample.get("continuation"), str):
                raise ValueError("Every generation sample must include its continuation.")

    def summary(self):
        metrics = self.evaluation["metrics"]
        dataset = self.evaluation["dataset"]
        initial_loss = metrics["initial_validation_loss"]
        best_loss = metrics["best_validation_loss"]
        return {
            "initial_validation_loss": initial_loss,
            "best_validation_loss": best_loss,
            "validation_perplexity": metrics["validation_perplexity"],
            "loss_reduction": initial_loss - best_loss,
            "validation_documents": dataset["validation_documents"],
            "validation_windows": dataset["validation_windows"],
            "validation_target_tokens": dataset["validation_target_tokens"],
            "generation_samples": len(self.generations),
            "tokenizer_sha256": self.evaluation["tokenizer_sha256"],
        }

    @staticmethod
    def _ngrams(tokens, n):
        return [
            tuple(tokens[index:index + n])
            for index in range(len(tokens) - n + 1)
        ]

    def generation_indicators(self):
        continuations = [
            sample["continuation"].lower().split()
            for sample in self.generations
        ]
        results = {
            "sample_count": len(continuations),
            "mean_continuation_words": (
                sum(len(tokens) for tokens in continuations) / len(continuations)
            ),
        }
        for n in (1, 2, 3):
            grams = [
                gram
                for tokens in continuations
                for gram in self._ngrams(tokens, n)
            ]
            results[f"distinct_{n}"] = (
                len(set(grams)) / len(grams) if grams else 0.0
            )
        trigrams = [
            gram
            for tokens in continuations
            for gram in self._ngrams(tokens, 3)
        ]
        results["repeated_trigram_fraction"] = (
            1.0 - len(set(trigrams)) / len(trigrams) if trigrams else 0.0
        )
        return results

    def save_generation_indicators(self, output_path):
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(
            json.dumps(self.generation_indicators(), ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        return output_path

## 3. Read the held-out scores

Cross-entropy and perplexity summarize next-token prediction over the validation targets. Loss reduction compares the best saved checkpoint with the model before training. These values are meaningful only alongside the tokenizer, split, context length, and data size recorded in the report.

In [ ]:
report = ModelEvaluationReport(EVALUATION_PATH, GENERATIONS_PATH)
summary = report.summary()
for name, value in summary.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

## 4. Inspect generated continuations

Read the prompts and outputs directly. Automatic diversity measures can flag repeated wording, but they cannot judge grammar, spelling conventions, code-switching, meaning, or cultural appropriateness.

In [ ]:
for number, sample in enumerate(report.generations, start=1):
    print(f"Sample {number}")
    print("Prompt:", sample["prompt"])
    print("Continuation:", sample["continuation"])
    print()

## 5. Calculate simple diversity indicators

Distinct-n is the fraction of unique word n-grams among all generated n-grams. The repeated-trigram fraction is the complement of distinct-3. Higher diversity is not automatically better: an incoherent output can also have varied words.

In [ ]:
generation_indicators = report.generation_indicators()
for name, value in generation_indicators.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

indicators_path = report.save_generation_indicators(
    EVALUATION_DIR / "generation_indicators.json"
)
print("Saved:", indicators_path.relative_to(PROJECT_ROOT))

## Evaluation scope

This report combines a quantitative held-out language-model score with a small qualitative sample review. It does not claim that perplexity or n-gram diversity measures Darija quality on their own. For a more reliable assessment, grow a carefully checked test set, record spelling and dialect conventions, and review the same prompts across model versions.